# Advanced Problems: Decorators, Parameterized Decorators, and Decorator Classes

This notebook contains advanced practice problems with full solutions.

Topics covered:

- decorator factories
- parameterized decorators
- `functools.wraps`
- callable class decorators
- stateful decorators
- stacked decorators
- timing decorators
- preserving metadata
- handling `*args` and `**kwargs`

## Problem 1 — Build a Parameterized Timing Decorator

Create a decorator factory called `timed(num_reps=1)`.

Requirements:

- It should return a decorator.
- The decorator should run the decorated function `num_reps` times.
- It should print the average runtime.
- It should return the final result.
- It must preserve the original function metadata using `functools.wraps`.

In [1]:
from functools import wraps
from time import perf_counter


def timed(num_reps=1):
    if not isinstance(num_reps, int) or num_reps < 1:
        raise ValueError('num_reps must be a positive integer')

    def decorator(fn):
        @wraps(fn)
        def inner(*args, **kwargs):
            total_elapsed = 0
            result = None

            for _ in range(num_reps):
                start = perf_counter()
                result = fn(*args, **kwargs)
                end = perf_counter()
                total_elapsed += end - start

            avg_elapsed = total_elapsed / num_reps
            print(f'Avg runtime: {avg_elapsed:.6f}s over {num_reps} reps')
            return result

        return inner

    return decorator


@timed(5)
def add(a, b):
    """Return the sum of two numbers."""
    return a + b


print(add(10, 20))
print(add.__name__)
print(add.__doc__)

Avg runtime: 0.000001s over 5 reps
30
add
Return the sum of two numbers.


## Problem 2 — Explain What Python Does with `@decorator_factory(...)`

Given this syntax:

```python
@decorator_factory(10, 20)
def func():
    pass
```

Rewrite it using the long-form assignment syntax.

In [2]:
# Solution:

def decorator_factory(a, b):
    def decorator(fn):
        @wraps(fn)
        def inner(*args, **kwargs):
            print(f'a={a}, b={b}')
            return fn(*args, **kwargs)
        return inner
    return decorator


def func():
    print('running func')


# This is what @decorator_factory(10, 20) does:
func = decorator_factory(10, 20)(func)

func()

a=10, b=20
running func


## Problem 3 — Build a Call Counter Decorator

Create a decorator called `count_calls`.

Requirements:

- Count how many times the decorated function is called.
- Store the count on the wrapper as `call_count`.
- Preserve function metadata.
- Work with any positional and keyword arguments.

In [3]:
def count_calls(fn):
    @wraps(fn)
    def inner(*args, **kwargs):
        inner.call_count += 1
        return fn(*args, **kwargs)

    inner.call_count = 0
    return inner


@count_calls
def greet(name, punctuation='!'):
    return f'Hello, {name}{punctuation}'


print(greet('Python'))
print(greet('Decorators', punctuation='!!!'))
print(greet.call_count)

Hello, Python!
Hello, Decorators!!!
2


## Problem 4 — Build a Parameterized Logging Decorator

Create a decorator factory called `logged(prefix)`.

Requirements:

- Print the prefix before calling the function.
- Print the function name.
- Print `args` and `kwargs`.
- Return the function result.
- Preserve metadata.

In [4]:
def logged(prefix):
    def decorator(fn):
        @wraps(fn)
        def inner(*args, **kwargs):
            print(f'{prefix} calling {fn.__name__}')
            print(f'args={args}')
            print(f'kwargs={kwargs}')
            result = fn(*args, **kwargs)
            print(f'{prefix} result={result}')
            return result
        return inner
    return decorator


@logged('[DEBUG]')
def multiply(a, b=1):
    return a * b


multiply(10, b=5)

[DEBUG] calling multiply
args=(10,)
kwargs={'b': 5}
[DEBUG] result=50


50

## Problem 5 — Build a Callable Class Decorator

Create a class called `Repeat` that can be used as a decorator.

Example:

```python
@Repeat(3)
def say_hi():
    print('hi')
```

Requirements:

- `Repeat(n)` should create a callable decorator object.
- The decorated function should run `n` times.
- The final result should be returned.
- Metadata should be preserved.

In [5]:
class Repeat:
    def __init__(self, n):
        if not isinstance(n, int) or n < 1:
            raise ValueError('n must be a positive integer')
        self.n = n

    def __call__(self, fn):
        @wraps(fn)
        def inner(*args, **kwargs):
            result = None
            for _ in range(self.n):
                result = fn(*args, **kwargs)
            return result
        return inner


@Repeat(3)
def say_hi(name):
    print(f'Hi, {name}')
    return name.upper()


print(say_hi('Guido'))
print(say_hi.__name__)

Hi, Guido
Hi, Guido
Hi, Guido
GUIDO
say_hi


## Problem 6 — Build a Stateful Callable Class Decorator

Create a class decorator called `CallStats`.

Requirements:

- It should be used as `@CallStats()`.
- It should count calls.
- It should store the last return value.
- It should expose both as attributes on the decorated function wrapper:
  - `call_count`
  - `last_result`

In [6]:
class CallStats:
    def __call__(self, fn):
        @wraps(fn)
        def inner(*args, **kwargs):
            inner.call_count += 1
            inner.last_result = fn(*args, **kwargs)
            return inner.last_result

        inner.call_count = 0
        inner.last_result = None
        return inner


@CallStats()
def power(base, exponent=2):
    return base ** exponent


print(power(3))
print(power(2, exponent=5))
print(power.call_count)
print(power.last_result)

9
32
2
32


## Problem 7 — Compose Multiple Decorators Correctly

Use both `@count_calls` and `@timed(3)` on the same function.

Then answer:

- Which decorator is applied first?
- Which wrapper runs first when the function is called?
- What happens to `call_count`?

In [7]:
@count_calls
@timed(3)
def slow_add(a, b):
    total = 0
    for i in range(100_000):
        total += i
    return a + b


print(slow_add(10, 20))
print(slow_add.call_count)


# Explanation:
#
# Decorators are applied bottom-up:
#
# slow_add = count_calls(timed(3)(slow_add))
#
# So @timed(3) is applied first.
# But when calling slow_add(), the outer wrapper from count_calls runs first.
# call_count increments once per external call to slow_add(), not once per timing repetition.

Avg runtime: 0.006111s over 3 reps
30
1


## Problem 8 — Create a Retry Decorator Factory

Create a decorator factory called `retry(max_attempts, exceptions)`.

Requirements:

- Retry the function if it raises one of the specified exceptions.
- Stop after `max_attempts`.
- Re-raise the last exception if all attempts fail.
- Preserve metadata.
- Validate that `max_attempts` is at least 1.

In [8]:
def retry(max_attempts=3, exceptions=(Exception,)):
    if not isinstance(max_attempts, int) or max_attempts < 1:
        raise ValueError('max_attempts must be a positive integer')

    def decorator(fn):
        @wraps(fn)
        def inner(*args, **kwargs):
            last_error = None

            for attempt in range(1, max_attempts + 1):
                try:
                    return fn(*args, **kwargs)
                except exceptions as ex:
                    last_error = ex
                    print(f'Attempt {attempt} failed: {ex}')

            raise last_error

        return inner

    return decorator


attempts = {'count': 0}


@retry(max_attempts=4, exceptions=(ValueError,))
def unstable():
    attempts['count'] += 1
    if attempts['count'] < 3:
        raise ValueError('not ready yet')
    return 'success'


print(unstable())

Attempt 1 failed: not ready yet
Attempt 2 failed: not ready yet
success


## Problem 9 — Decorator Class with Configuration and State

Create a callable class decorator called `LimitCalls`.

Example:

```python
@LimitCalls(max_calls=3)
def func():
    pass
```

Requirements:

- Allow only `max_calls` successful calls.
- After that, raise `RuntimeError`.
- Keep the current count as state.
- Preserve metadata.

In [9]:
class LimitCalls:
    def __init__(self, max_calls):
        if not isinstance(max_calls, int) or max_calls < 1:
            raise ValueError('max_calls must be a positive integer')
        self.max_calls = max_calls
        self.count = 0

    def __call__(self, fn):
        @wraps(fn)
        def inner(*args, **kwargs):
            if self.count >= self.max_calls:
                raise RuntimeError(f'{fn.__name__} exceeded {self.max_calls} calls')

            self.count += 1
            return fn(*args, **kwargs)

        return inner


@LimitCalls(max_calls=3)
def limited_greet(name):
    return f'Hello, {name}'


print(limited_greet('A'))
print(limited_greet('B'))
print(limited_greet('C'))

try:
    print(limited_greet('D'))
except RuntimeError as ex:
    print(type(ex).__name__, ex)

Hello, A
Hello, B
Hello, C
RuntimeError limited_greet exceeded 3 calls


## Problem 10 — Best-Practice Flexible Decorator

Create a decorator called `trace` that supports both forms:

```python
@trace
def f():
    pass
```

and:

```python
@trace(prefix='DEBUG')
def f():
    pass
```

Hint: The decorator must detect whether it was called with a function or with configuration arguments.

In [10]:
def trace(fn=None, *, prefix='TRACE'):
    def decorator(func):
        @wraps(func)
        def inner(*args, **kwargs):
            print(f'{prefix}: calling {func.__name__}')
            return func(*args, **kwargs)
        return inner

    if fn is None:
        return decorator

    return decorator(fn)


@trace
def plain():
    return 'plain result'


@trace(prefix='DEBUG')
def configured():
    return 'configured result'


print(plain())
print(configured())

TRACE: calling plain
plain result
DEBUG: calling configured
configured result


# Summary

Key ideas:

- A basic decorator receives a function and returns a replacement callable.
- A parameterized decorator is usually a decorator factory.
- `@decorator(args)` means Python first calls `decorator(args)` and then applies the returned decorator.
- Callable classes can also be decorators by implementing `__call__`.
- Use `functools.wraps` to preserve metadata.
- Stateful decorators can store state in closures, wrapper attributes, or class instances.
- Decorators are applied bottom-up but wrappers execute top-down.